# Predicción de tiempos de entrega y anticipación de incumplimientos de SLA — Modelado (SERVIENVIOS, Bogotá)

Universidad Santo Tomás · Facultad de Estadística · Maestría en Ciencia de Datos

John Fernando Cruz Becerra · Oscar Javier Huertas Campo · Mauricio Alejandro Gaviria Alzate

Este notebook parte de la **base analítica definitiva** ("capa Gold") que construye `0300_CleanTransformParquet_1.ipynb` y homologa, sobre esos datos reales de SERVIENVIOS, el mismo patrón de trabajo que `0400_Proyecto_EJEMPLO_1.ipynb` (celda 28 en adelante) usa sobre su dataset de ejemplo: carga de la capa Gold, validación, análisis descriptivo, verificación de balance de clases, preparación del dataset de ML, partición temporal, entrenamiento y comparación de modelos, evaluación e interpretabilidad.

**Diferencia deliberada frente al ejemplo**: el ejemplo (`0400_Proyecto_EJEMPLO_1.ipynb`) usa PySpark/Databricks ML (`StringIndexer`, `VectorAssembler`, `Pipeline` de `pyspark.ml`, Regresión Logística y Random Forest). El anteproyecto real (sección 4.8, "Herramientas") especifica explícitamente **Python, pandas, NumPy, scikit-learn, XGBoost, LightGBM, SHAP y matplotlib/seaborn** — no Spark ni Databricks ML — así que este notebook homologa la *estructura* del ejemplo (Gold → EDA → balance de clases → dataset ML → partición temporal → entrenamiento → evaluación → comparación → conclusiones) pero con ese stack, que es el que efectivamente exige el documento de proyecto. La construcción de la capa Gold en sí (`0300`) sí corre en Databricks, porque esa es la infraestructura de datos que ya adoptó el equipo; este notebook puede ejecutarse dentro de Databricks (leyendo el mismo Volume) o en Colab (si se copia la base Gold a Drive), gracias a la celda de carga con respaldo automático de abajo.

**Honestidad ante todo**: este notebook se construyó y probó de punta a punta contra datos **sintéticos** que replican el esquema exacto de la base Gold (no contra los datos reales de SERVIENVIOS, a los que este entorno no tiene acceso). Las cifras y gráficos que produzca la primera vez que el equipo lo corra con datos reales serán distintas; lo que está validado es que el código corre sin errores de wiring (nombres de columna, tipos, orden de operaciones) sobre ese esquema. Al final se listan explícitamente los supuestos que el equipo debe confirmar con datos reales.


> ### 🔧 Registro de la refactorización
> Esta versión corrige los errores que aparecían al ejecutar el notebook en Databricks:
>
> 1. **`NameError: resultados_regresion`** – la celda de regresión (sección 5) redefinía `TARGET_REGRESION = "TU_COLUMNA_TARGET"`, `TIENE_XGBOOST/LIGHTGBM = False` y `FEATURES_REGRESION` con columnas que no existen, y además tenía la ejecución comentada. Ahora usa las variables definidas en la sección 4 y sí entrena y evalúa.
> 2. **`incumplimiento_sla` no es binario** – en la base Gold real toma muchos valores (0, 1, 2, 3, … 34 distintos; es un conteo de días de retraso), por lo que la clasificación (`precision_score`, `roc_auc_score`…) habría fallado como multiclase. Se crea `incumplimiento_sla_bin = (incumplimiento_sla > 0)`.
> 3. **Nombres de columnas incorrectos** – `SEG_NUMERO_PIEZAS` / `SEG_PESO_ENVIO` no existen; las reales son `TOTAL_PIEZAS` / `TOTAL_PESO_ENVIO`.
> 4. **Categóricas mal detectadas** – `CATEGORIA_ZONA_URBA`, `DIA_SEMANA_ENVIO` y `G_ID_TIPOGUIA` son numéricas en el parquet pero son categorías; ahora se castean a texto y se usa la lista explícita `CATEGORICAS`.
> 5. **Fuga de información (target leakage) en train** – el target encoding y el promedio por zona+franja ahora se calculan *out-of-fold* en train.
> 6. **`MES_ENVIO` retirada de los features** – con partición temporal, train cubre ene–jun y test jul–ago: el modelo nunca vio esos meses.
> 7. **Métricas** – MAPE ignora entregas < 1 h (explotaba con `y_true ≈ 0`), se añade WAPE, las predicciones se recortan a ≥ 0 y se agrega un baseline ingenuo.
> 8. **Rendimiento / robustez** – preprocesador clonado por modelo, regresión con XGBoost y LightGBM (antes solo existían en clasificación), permutación sobre muestra, SHAP con muestra aleatoria y compatible con `shap >= 0.45` (salidas 3D).

---
## 0. Carga de la base analítica definitiva (capa Gold)

Intenta primero la ruta de Databricks (la misma que usa `0300_CleanTransformParquet_1.ipynb`); si `dbutils` no existe en este entorno (p. ej. si el notebook se corre en Colab), recurre a una carpeta de Drive como respaldo, siguiendo el mismo patrón de resolución automática que ya usa `0200_AnalisisExploratorio.ipynb`.

In [0]:
# !pip install duckdb
# !pip install seaborn
# !pip install holidays
# !pip install xgboost
# !pip install lightgbm
# !pip install shap


In [0]:
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix,
)
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold

# XGBoost, LightGBM y SHAP: exigidos por el anteproyecto (sección 4.8), pero se importan con try/except
# para que el notebook no se caiga completo si el entorno de quien lo ejecute no los tiene instalados
# todavía -- en ese caso, esas dos columnas del comparativo simplemente se omiten y se avisa por qué.
try:
    import xgboost as xgb
    TIENE_XGBOOST = True
except ImportError:
    TIENE_XGBOOST = False
    print("AVISO: xgboost no está instalado en este entorno (`pip install xgboost`). Se omite XGBoost.")

try:
    import lightgbm as lgb
    TIENE_LIGHTGBM = True
except ImportError:
    TIENE_LIGHTGBM = False
    print("AVISO: lightgbm no está instalado en este entorno (`pip install lightgbm`). Se omite LightGBM.")

try:
    import shap
    TIENE_SHAP = True
except ImportError:
    TIENE_SHAP = False
    print("AVISO: shap no está instalado en este entorno (`pip install shap`). Se omite la interpretabilidad SHAP "
          "y se deja solo la importancia por permutación (que no depende de una librería externa).")

try:
    import holidays
    TIENE_HOLIDAYS = True
except ImportError:
    TIENE_HOLIDAYS = False
    print("AVISO: holidays no está instalado (`pip install holidays`). Se omite la variable 'es_festivo'.")

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [0]:
# 1. Resolución automática de la ruta de la base Gold: Databricks primero, Colab como respaldo.
try:
    dbutils  # noqa: solo existe dentro de un entorno Databricks
    PATH_BASE = "/Volumes/proyectogrado/source/sourcedb/"
    print("Entorno Databricks detectado.")
except NameError:
    # Ajusta esta ruta a tu carpeta real de Google Drive montada en Colab
    PATH_BASE = "/content/drive/MyDrive/proyectogrado/source/sourcedb/"
    print("Entorno Colab / Local detectado.")

RUTA_BASE_ANALITICA_DEFINITIVA = f"{PATH_BASE}LOG_CONSOLIDADO/BASE_ANALITICA_DEFINITIVA.parquet"
print("Leyendo la base Gold desde:", RUTA_BASE_ANALITICA_DEFINITIVA)

# 2. Consulta SQL limpia sin paréntesis sobrantes
query = f"""
    SELECT * 
    FROM read_parquet('{RUTA_BASE_ANALITICA_DEFINITIVA}') 
    ---WHERE G_NOMBRE_REGIONAL_ORIGEN = 'BOGOTA'
"""

df_gold = duckdb.sql(query).df()

print(f"Base Gold cargada: {len(df_gold):,} filas, {df_gold.shape[1]} columnas.")
df_gold.head()

## 1. Validación de la capa Gold (homologa la validación de conteos/Silver del ejemplo)

In [0]:
# Conteo, nulos en las variables objetivo, y distribución del split train/test que ya viene calculado
# desde 0300 (columna SPLIT, partición temporal 80/20 -- sección 4.6 del anteproyecto).
print("Filas por SPLIT:")
if "SPLIT" in df_gold.columns:
    display(df_gold["SPLIT"].value_counts())

# Columnas que esperamos analizar
cols_objetivo = [c for c in ["tiempo_entrega_horas", "incumplimiento_sla"] if c in df_gold.columns]

if cols_objetivo:
    print("\nNulos en las variables objetivo presentes:")
    display(df_gold[cols_objetivo].isna().sum())
else:
    print("\n⚠️ Advertencia: Las columnas 'tiempo_entrega_horas' e 'incumplimiento_sla' aún no existen en df_gold.")
    print("Columnas disponibles actualmente:", list(df_gold.columns))

print("\nGuías cuyo incumplimiento se calculó con SLA_DIAS_DEFAULT aproximado (SLA_ES_APROXIMADO=True):")
if "SLA_ES_APROXIMADO" in df_gold.columns:
    display(df_gold["SLA_ES_APROXIMADO"].value_counts(normalize=True))

### 1.1 Diagnóstico y binarización de `incumplimiento_sla`

En la base Gold, `incumplimiento_sla` **no es una bandera 0/1**: toma muchos valores enteros (en las filas de ejemplo, p. ej., 262 h con SLA de 24 h → 4; 38 h → 0), es decir, se comporta como *días de retraso* respecto al compromiso. Para el problema de clasificación se define la etiqueta binaria `incumplimiento_sla_bin = 1` cuando hay algún retraso.

> ⚠️ **Confirmar con `0300`**: el umbral `UMBRAL_DIAS_INCUMPLIMIENTO = 0` asume que "> 0 días de retraso" equivale a incumplir el SLA. Si el criterio del anteproyecto es distinto (p. ej. tolerancia de 1 día), basta cambiar esa constante.

In [0]:
print("Valores distintos de incumplimiento_sla:", df_gold["incumplimiento_sla"].nunique())
display(
    df_gold["incumplimiento_sla"].value_counts(normalize=True).sort_index().head(10)
    .rename("proporción").to_frame()
)

UMBRAL_DIAS_INCUMPLIMIENTO = 0   # > 0 días de retraso => incumple
df_gold["incumplimiento_sla_bin"] = (df_gold["incumplimiento_sla"] > UMBRAL_DIAS_INCUMPLIMIENTO).astype(int)

print("\nProporción de la etiqueta binaria (0 = cumple, 1 = incumple):")
display(df_gold["incumplimiento_sla_bin"].value_counts(normalize=True).sort_index().rename("proporción").to_frame())

In [0]:
# Distribución de la variable objetivo continua (homologa el análisis descriptivo de la capa Gold del ejemplo)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_gold["tiempo_entrega_horas"], bins=40, ax=axes[0])
axes[0].set_title("Distribución del tiempo de entrega (horas)")
sns.boxplot(x=df_gold["tiempo_entrega_horas"], ax=axes[1])
axes[1].set_title("Boxplot del tiempo de entrega (horas)")
plt.tight_layout()
plt.show()

## 2. Análisis por zona/regional y tendencia temporal (homologa el análisis de rentabilidad por zona y la tendencia de ventas del ejemplo, adaptado a tiempos de entrega e incumplimiento)

In [0]:
df_por_regional = duckdb.sql("""
    SELECT
        G_NOMBRE_REGIONAL_DESTINO,
        CATEGORIA_ZONA_URBA,
        COUNT(1) AS TOTAL_ENVIOS,
        AVG(tiempo_entrega_horas) AS TIEMPO_PROMEDIO_HORAS,
        AVG(incumplimiento_sla_bin) AS TASA_INCUMPLIMIENTO,
        AVG(incumplimiento_sla) AS DIAS_RETRASO_PROMEDIO
    FROM df_gold
    GROUP BY G_NOMBRE_REGIONAL_DESTINO, CATEGORIA_ZONA_URBA
    ORDER BY TASA_INCUMPLIMIENTO DESC
""").df()
display(df_por_regional)

In [0]:
df_tendencia_mensual = duckdb.sql("""
    SELECT
        MES_ENVIO,
        COUNT(1) AS TOTAL_ENVIOS,
        AVG(tiempo_entrega_horas) AS TIEMPO_PROMEDIO_HORAS,
        AVG(incumplimiento_sla_bin) AS TASA_INCUMPLIMIENTO,
        AVG(incumplimiento_sla) AS DIAS_RETRASO_PROMEDIO
    FROM df_gold
    GROUP BY MES_ENVIO
    ORDER BY MES_ENVIO
""").df()
display(df_tendencia_mensual)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(df_tendencia_mensual["MES_ENVIO"], df_tendencia_mensual["TIEMPO_PROMEDIO_HORAS"], marker="o", color="tab:blue")
ax1.set_ylabel("Tiempo promedio de entrega (horas)", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(df_tendencia_mensual["MES_ENVIO"], df_tendencia_mensual["TASA_INCUMPLIMIENTO"], marker="s", color="tab:red")
ax2.set_ylabel("Tasa de incumplimiento SLA", color="tab:red")
ax1.set_xlabel("Mes de envío")
plt.title("Tendencia mensual: tiempo de entrega e incumplimiento de SLA")
plt.tight_layout()
plt.show()

## 3. Balance de clases de `incumplimiento_sla` (homologa la verificación de balance del ejemplo)

In [0]:
# Balance de clases sobre la etiqueta BINARIA (0 = cumple, 1 = incumple)
df_conteo = (
    df_gold["incumplimiento_sla_bin"].value_counts().sort_index()
    .rename_axis("estado").reset_index(name="conteo")
)

plt.figure(figsize=(6, 4.5))
ax = sns.barplot(
    data=df_conteo, x="estado", y="conteo", hue="estado",
    palette=["#2b5c8f", "#d9534f"], legend=False,   # Azul = Cumple, Rojo = Incumple
)
plt.title("Distribución de Cumplimiento (0) vs. Incumplimiento (1) de SLA", fontsize=11, pad=15)
plt.xlabel("Estado SLA", fontsize=10)
plt.ylabel("Cantidad de Guías", fontsize=10)
plt.xticks([0, 1], ["Cumple (0)", "Incumple (1)"])
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

total_guias = df_conteo["conteo"].sum()
for p in ax.patches:
    altura = p.get_height()
    if altura > 0:
        ax.annotate(
            f"{int(altura):,}\n({100 * altura / total_guias:.1f}%)",
            (p.get_x() + p.get_width() / 2.0, altura),
            ha="center", va="bottom", fontsize=9.5, weight="bold",
            xytext=(0, 5), textcoords="offset points",
        )
plt.ylim(0, df_conteo["conteo"].max() * 1.18)
sns.despine()
plt.tight_layout()
plt.show()

## 4. Preparación del dataset de aprendizaje automático

Ingeniería de características adicional (sección 4.3.3) que la base Gold de `0300` todavía no incluye: indicador de día festivo/fin de semana, e interacción zona × franja horaria. Se construye aquí, sobre `df_gold`, antes de la partición (son deterministas por fila, no dependen de ningún agregado de train, así que no hay riesgo de fuga de información en este paso).

In [0]:
df_gold["G_FECHA_ENVIO"] = pd.to_datetime(df_gold["G_FECHA_ENVIO"])
df_gold["es_fin_de_semana"] = df_gold["G_FECHA_ENVIO"].dt.weekday.isin([5, 6]).astype(int)

if TIENE_HOLIDAYS:
    co_holidays = holidays.CO(years=range(df_gold["G_FECHA_ENVIO"].dt.year.min(), df_gold["G_FECHA_ENVIO"].dt.year.max() + 2))
    df_gold["es_festivo"] = df_gold["G_FECHA_ENVIO"].dt.date.isin(co_holidays).astype(int)
else:
    df_gold["es_festivo"] = 0

df_gold["zona_franja"] = df_gold["CATEGORIA_ZONA_URBA"].astype(str) + "_" + df_gold["FRANJA_HORARIA_ENVIO"].astype(str)

df_gold[["G_FECHA_ENVIO", "es_fin_de_semana", "es_festivo", "zona_franja"]].head()

In [0]:
# Partición train/test: se reutiliza la columna SPLIT que ya calculó 0300 (partición temporal 80/20,
# sección 4.6), para que la base Gold sea la única fuente de verdad del corte train/test.
df_train = df_gold[df_gold["SPLIT"] == "train"].copy().reset_index(drop=True)
df_test = df_gold[df_gold["SPLIT"] == "test"].copy().reset_index(drop=True)
print(f"Train: {len(df_train):,} filas | Test: {len(df_test):,} filas")

# ---------------------------------------------------------------- objetivos
TARGET_REGRESION = "tiempo_entrega_horas"
TARGET_CLASIFICACION = "incumplimiento_sla_bin"      # binaria, creada en la sección 1.1

# ---------------------------------------------------------------- variables
# Categóricas (algunas vienen numéricas en el parquet pero son categorías: zona 1.0-8.0, día 1-7, tipo de guía).
CATEGORICAS = ["G_NOMBRE_REGIONAL_ORIGEN", "G_NOMBRE_REGIONAL_DESTINO", "CATEGORIA_ZONA_URBA",
               "FRANJA_HORARIA_ENVIO", "DIA_SEMANA_ENVIO", "G_ID_TIPOGUIA"]
# MES_ENVIO se excluye a propósito: con partición temporal train = ene-jun y test = jul-ago, así que el
# modelo nunca vería en entrenamiento los meses que debe predecir (solo captaría deriva, no señal).
NUMERICAS_BASE = ["TOTAL_PIEZAS", "TOTAL_PESO_ENVIO", "VOLUMEN_ENVIO_CM3",
                  "TIEMPO_ENTREGA_COMERCIAL_HOR", "es_fin_de_semana", "es_festivo",
                  "tiempo_promedio_zona_franja"]

for df_ in (df_train, df_test):
    for c in CATEGORICAS:
        df_[c] = df_[c].astype(str)           # evita que 1.0, 2.0... se traten como numéricas

# ---------------------------------------------------------------- target encoding SIN fuga (out-of-fold)
def target_encode_oof(df_tr, df_te, columna, objetivo, suavizado=10, n_splits=5, seed=RANDOM_STATE):
    '''Target encoding con suavizado bayesiano.
    - Train: codificación out-of-fold (cada fila se codifica con estadísticas de OTROS folds de train;
      así la fila no "ve" su propio objetivo).
    - Test: se codifica con estadísticas de TODO train.'''
    y = df_tr[objetivo].astype(float)

    def _mapa(cats, yy):
        g = yy.groupby(cats).agg(["mean", "count"])
        media = yy.mean()
        return (g["mean"] * g["count"] + media * suavizado) / (g["count"] + suavizado), media

    enc_train = np.empty(len(df_tr), dtype="float32")
    for idx_fit, idx_apply in KFold(n_splits, shuffle=True, random_state=seed).split(df_tr):
        mapa, media = _mapa(df_tr[columna].iloc[idx_fit], y.iloc[idx_fit])
        enc_train[idx_apply] = df_tr[columna].iloc[idx_apply].map(mapa).fillna(media).to_numpy()

    mapa, media = _mapa(df_tr[columna], y)
    enc_test = df_te[columna].map(mapa).fillna(media).to_numpy(dtype="float32")
    return enc_train, enc_test

# Agregado histórico de tiempo por zona+franja (sección 4.3.3)
df_train["tiempo_promedio_zona_franja"], df_test["tiempo_promedio_zona_franja"] = target_encode_oof(
    df_train, df_test, "zona_franja", TARGET_REGRESION, suavizado=5)

# Ciudad destino (alta cardinalidad): una codificación por tarea, contra su propio objetivo
df_train["ciudad_destino_enc_reg"], df_test["ciudad_destino_enc_reg"] = target_encode_oof(
    df_train, df_test, "G_NOMBRE_CIUDAD_DESTINO", TARGET_REGRESION)
df_train["ciudad_destino_enc_clf"], df_test["ciudad_destino_enc_clf"] = target_encode_oof(
    df_train, df_test, "G_NOMBRE_CIUDAD_DESTINO", TARGET_CLASIFICACION)

FEATURES_REGRESION = CATEGORICAS + NUMERICAS_BASE + ["ciudad_destino_enc_reg"]
FEATURES_CLASIFICACION = CATEGORICAS + NUMERICAS_BASE + ["ciudad_destino_enc_clf"]

# tiempo_entrega_horas e incumplimiento_sla NUNCA entran como feature de clasificación (definen la etiqueta).
numericas_modelo = [c for c in set(NUMERICAS_BASE + ["ciudad_destino_enc_reg", "ciudad_destino_enc_clf"])]
for df_ in (df_train, df_test):
    df_[numericas_modelo] = df_[numericas_modelo].apply(pd.to_numeric, errors="coerce").astype("float32")

# Verificación temprana de que todas las columnas existen (falla aquí con un mensaje claro, no más adelante)
faltantes = [c for c in set(FEATURES_REGRESION + FEATURES_CLASIFICACION + [TARGET_REGRESION, TARGET_CLASIFICACION])
             if c not in df_train.columns]
assert not faltantes, f"Faltan columnas en df_train: {faltantes}"

# TODO (pendiente, sección 4.3.2): variables 'de cliente' (tipo_cliente, prioridad/criticidad, acuerdo de
# nivel de servicio aplicable) todavía no están mapeadas desde LOG.GUIAS en 0300.
print("FEATURES_REGRESION:", FEATURES_REGRESION)
print("FEATURES_CLASIFICACION:", FEATURES_CLASIFICACION)

---
## 5. Etapa 3 — Modelado de regresión (tiempo de entrega)

Objetivo específico 3: comparar regresión lineal (línea base), Random Forest, XGBoost y LightGBM para predecir `tiempo_entrega_horas`, con validación temporal y métricas MAE, RMSE y MAPE (sección 4.7).

In [0]:
# NOTA: aquí NO se redefine TARGET_REGRESION, RANDOM_STATE, TIENE_XGBOOST/LIGHTGBM ni FEATURES_REGRESION:
# vienen de las secciones 0 y 4. (La versión anterior los pisaba con valores de plantilla.)

def construir_preprocesador(features, categoricas):
    numericas = [f for f in features if f not in categoricas]
    pipeline_num = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    pipeline_cat = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
    ])
    return ColumnTransformer(
        transformers=[("cat", pipeline_cat, categoricas), ("num", pipeline_num, numericas)],
        sparse_threshold=0,
    )

def evaluar_regresion(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)          # un tiempo de entrega no puede ser negativo
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true >= 1                                                # MAPE solo con entregas >= 1 h (evita dividir por ~0)
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    wape = np.abs(y_true - y_pred).sum() / np.abs(y_true).sum() * 100  # error ponderado, robusto a valores pequeños
    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape, "WAPE_%": wape}

def modelos_regresion():
    modelos = {
        "Regresión lineal (línea base)": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=100, max_depth=12, min_samples_leaf=20, max_samples=0.5,
            random_state=RANDOM_STATE, n_jobs=-1,
        ),
    }
    if TIENE_XGBOOST:
        modelos["XGBoost"] = xgb.XGBRegressor(
            n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
        )
    if TIENE_LIGHTGBM:
        modelos["LightGBM"] = lgb.LGBMRegressor(
            n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
        )
    return modelos

def entrenar_y_evaluar_regresion(features):
    X_train, y_train = df_train[features], df_train[TARGET_REGRESION]
    X_test, y_test = df_test[features], df_test[TARGET_REGRESION]
    categoricas = [c for c in CATEGORICAS if c in features]
    preprocesador = construir_preprocesador(features, categoricas)

    # Baseline ingenuo: predecir el promedio histórico de la zona+franja (referencia mínima a superar, H1)
    resultados = [{**evaluar_regresion(y_test, df_test["tiempo_promedio_zona_franja"]),
                   "modelo": "Baseline ingenuo (promedio zona+franja)"}]
    pipelines = {}
    for nombre, modelo in modelos_regresion().items():
        print(f"Entrenando modelo: {nombre}...")
        pipe = Pipeline([("prep", clone(preprocesador)), ("modelo", modelo)])   # clone: cada modelo ajusta su propio preprocesador
        pipe.fit(X_train, y_train)
        resultados.append({**evaluar_regresion(y_test, pipe.predict(X_test)), "modelo": nombre})
        pipelines[nombre] = pipe

    return pd.DataFrame(resultados)[["modelo", "MAE", "RMSE", "MAPE_%", "WAPE_%"]], pipelines

resultados_regresion, pipelines_regresion = entrenar_y_evaluar_regresion(FEATURES_REGRESION)
display(resultados_regresion.sort_values("MAE"))

In [0]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=resultados_regresion, x="modelo", y="MAE", ax=ax)
ax.set_title("MAE por modelo (regresión — tiempo de entrega)")
ax.set_ylabel("MAE (horas)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

MEJOR_MODELO_REGRESION = (
    resultados_regresion[resultados_regresion["modelo"].isin(pipelines_regresion)]
    .sort_values("MAE").iloc[0]["modelo"]
)
print(f"Mejor modelo de regresión por MAE: {MEJOR_MODELO_REGRESION}")
print(
    "Nota (H1): para que el anteproyecto considere validada la hipótesis principal, el mejor modelo de "
    "ensamble debe superar de manera significativa a la regresión lineal de línea base -- compararlo "
    "explícitamente aquí una vez se corra con datos reales."
)

## 6. Etapa 4 — Modelado de clasificación (incumplimiento de SLA)

Objetivo específico 4: comparar regresión logística (línea base), Random Forest, XGBoost y LightGBM para predecir `incumplimiento_sla`, manejando el desbalance de clases y priorizando recall, con precisión, recall, F1, AUC-ROC y AUC-PR (sección 4.7).

In [0]:
def evaluar_clasificacion(y_true, y_pred, y_proba):
    return {
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "AUC-ROC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "AUC-PR": average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
    }

def modelos_clasificacion():
    modelos = {
        "Regresión logística (línea base)": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(
            n_estimators=200, max_depth=14, min_samples_leaf=50, max_samples=0.3,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
        ),
    }
    if TIENE_XGBOOST:
        n_pos = (df_train[TARGET_CLASIFICACION] == 1).sum()
        n_neg = (df_train[TARGET_CLASIFICACION] == 0).sum()
        modelos["XGBoost"] = xgb.XGBClassifier(
            n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=(n_neg / n_pos) if n_pos > 0 else 1.0, tree_method="hist",
            random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss",
        )
    if TIENE_LIGHTGBM:
        modelos["LightGBM"] = lgb.LGBMClassifier(
            n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
        )
    return modelos

def entrenar_y_evaluar_clasificacion(features):
    train_c = df_train.dropna(subset=[TARGET_CLASIFICACION])
    test_c = df_test.dropna(subset=[TARGET_CLASIFICACION])
    X_train, y_train = train_c[features], train_c[TARGET_CLASIFICACION].astype(int)
    X_test, y_test = test_c[features], test_c[TARGET_CLASIFICACION].astype(int)
    categoricas = [c for c in CATEGORICAS if c in features]
    preprocesador = construir_preprocesador(features, categoricas)

    resultados, pipelines = [], {}
    for nombre, modelo in modelos_clasificacion().items():
        print(f"Entrenando modelo: {nombre}...")
        pipe = Pipeline([("prep", clone(preprocesador)), ("modelo", modelo)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        resultados.append({**evaluar_clasificacion(y_test, y_pred, y_proba), "modelo": nombre})
        pipelines[nombre] = pipe
    cols = ["modelo", "Precision", "Recall", "F1", "AUC-ROC", "AUC-PR"]
    return pd.DataFrame(resultados)[cols], pipelines

resultados_clasificacion, pipelines_clasificacion = entrenar_y_evaluar_clasificacion(FEATURES_CLASIFICACION)
display(resultados_clasificacion.sort_values("AUC-PR", ascending=False))

### 6.1 Matriz de confusión del mejor modelo (por AUC-PR, más informativo bajo desbalance — sección 4.7)

In [0]:
MEJOR_MODELO_CLASIFICACION = resultados_clasificacion.sort_values("AUC-PR", ascending=False).iloc[0]["modelo"]
pipe_ganador_clf = pipelines_clasificacion[MEJOR_MODELO_CLASIFICACION]
print(f"Mejor modelo de clasificación por AUC-PR: {MEJOR_MODELO_CLASIFICACION}")

test_c = df_test.dropna(subset=[TARGET_CLASIFICACION])
X_test_c, y_test_c = test_c[FEATURES_CLASIFICACION], test_c[TARGET_CLASIFICACION].astype(int)
y_pred_c = pipe_ganador_clf.predict(X_test_c)

matriz = confusion_matrix(y_test_c, y_pred_c)
fig, ax = plt.subplots(figsize=(4, 4))
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Cumple (0)", "Incumple (1)"], yticklabels=["Cumple (0)", "Incumple (1)"], ax=ax)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de confusión — {MEJOR_MODELO_CLASIFICACION}")
plt.tight_layout()
plt.show()

print(
    "Un falso negativo (incumple pero el modelo predice que cumple) es más costoso operativamente que un "
    "falso positivo, según la asimetría de costos descrita en la sección 4.11 del anteproyecto -- si el "
    "recall de la clase 'incumple' es bajo, considerar bajar el umbral de decisión de 0.5 en vez de usar "
    "predict() directamente."
)

### 6.2 Intervalos de confianza por bootstrap (sección 4.7)

In [0]:
def bootstrap_ci(y_true, y_proba, metrica_fn, n_iter=200, alpha=0.05, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    valores = []
    for _ in range(n_iter):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_proba[idx]
        if len(np.unique(yt)) < 2:
            continue
        valores.append(metrica_fn(yt, yp))
    valores = np.array(valores)
    return np.mean(valores), np.percentile(valores, 100 * alpha / 2), np.percentile(valores, 100 * (1 - alpha / 2))

y_proba_c = pipe_ganador_clf.predict_proba(X_test_c)[:, 1]
media_auc, lo_auc, hi_auc = bootstrap_ci(y_test_c, y_proba_c, roc_auc_score)
media_pr, lo_pr, hi_pr = bootstrap_ci(y_test_c, y_proba_c, average_precision_score)
print(f"AUC-ROC: {media_auc:.3f}  IC95%: [{lo_auc:.3f}, {hi_auc:.3f}]")
print(f"AUC-PR:  {media_pr:.3f}  IC95%: [{lo_pr:.3f}, {hi_pr:.3f}]")

---
## 7. Etapa 5 — Interpretabilidad y hallazgos operativos

Objetivo específico 5: interpretar los modelos seleccionados mediante **valores SHAP** e **importancia por permutación**, identificando los factores de mayor peso predictivo (sección 4.13).

### 7.1 Importancia por permutación (agnóstica al modelo, siempre disponible)

In [0]:
def graficar_importancia_permutacion(pipe, X, y, titulo, scoring, top=15, n_muestra=20000, n_repeats=5):
    # Sobre ~380 mil filas x 10 repeticiones x cada variable sería innecesariamente lento: se usa una muestra.
    if len(X) > n_muestra:
        idx = X.sample(n_muestra, random_state=RANDOM_STATE).index
        X, y = X.loc[idx], y.loc[idx]
    resultado = permutation_importance(pipe, X, y, n_repeats=n_repeats, random_state=RANDOM_STATE,
                                       scoring=scoring, n_jobs=-1)
    orden = resultado.importances_mean.argsort()[::-1][:top]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh([X.columns[i] for i in orden][::-1], resultado.importances_mean[orden][::-1])
    ax.set_title(titulo)
    ax.set_xlabel("Caída media en el score al permutar la variable")
    plt.tight_layout()
    plt.show()
    return resultado

MEJOR_MODELO_REGRESION_PIPE = pipelines_regresion[MEJOR_MODELO_REGRESION]
_ = graficar_importancia_permutacion(
    MEJOR_MODELO_REGRESION_PIPE, df_test[FEATURES_REGRESION], df_test[TARGET_REGRESION],
    f"Importancia por permutación — regresión ({MEJOR_MODELO_REGRESION})", scoring="neg_mean_absolute_error",
)

_ = graficar_importancia_permutacion(
    pipe_ganador_clf, X_test_c, y_test_c,
    f"Importancia por permutación — clasificación ({MEJOR_MODELO_CLASIFICACION})", scoring="average_precision",
)

### 7.2 Valores SHAP (si la librería está disponible)

In [0]:
if TIENE_SHAP:
    def explicar_con_shap(pipe, X, titulo, max_muestras=500):
        prep = pipe.named_steps["prep"]
        modelo = pipe.named_steps["modelo"]
        Xm = X.sample(min(max_muestras, len(X)), random_state=RANDOM_STATE)   # muestra aleatoria (no las primeras filas)
        X_t = prep.transform(Xm)
        nombres_columnas = prep.get_feature_names_out()

        if isinstance(modelo, (LinearRegression, LogisticRegression)):
            valores_shap = shap.LinearExplainer(modelo, X_t).shap_values(X_t)
        else:
            valores_shap = shap.TreeExplainer(modelo).shap_values(X_t, check_additivity=False)

        if isinstance(valores_shap, list):          # versiones antiguas de shap: lista [clase0, clase1]
            valores_shap = valores_shap[-1]
        valores_shap = np.asarray(valores_shap)
        if valores_shap.ndim == 3:                  # versiones nuevas: (muestras, variables, clases)
            valores_shap = valores_shap[:, :, -1]

        shap.summary_plot(valores_shap, X_t, feature_names=nombres_columnas, show=False, max_display=15)
        plt.title(titulo)
        plt.tight_layout()
        plt.show()

    print("Interpretabilidad SHAP — regresión:")
    explicar_con_shap(MEJOR_MODELO_REGRESION_PIPE, df_test[FEATURES_REGRESION],
                      f"SHAP — regresión ({MEJOR_MODELO_REGRESION})")

    print("Interpretabilidad SHAP — clasificación:")
    explicar_con_shap(pipe_ganador_clf, X_test_c, f"SHAP — clasificación ({MEJOR_MODELO_CLASIFICACION})")
else:
    print("shap no está instalado en este entorno: se omite esta sección. "
          "La importancia por permutación de 7.1 ya da una lectura agnóstica al modelo mientras tanto.")

## 8. Comparación final de modelos

In [0]:
print("Regresión (tiempo de entrega):")
display(resultados_regresion.sort_values("MAE"))
print(f"Mejor: {MEJOR_MODELO_REGRESION}")

print("\nClasificación (incumplimiento de SLA):")
display(resultados_clasificacion.sort_values("AUC-PR", ascending=False))
print(f"Mejor: {MEJOR_MODELO_CLASIFICACION}")

## 9. Conclusiones y hallazgos operativos

*(Plantilla a completar por el equipo una vez el notebook corra con datos reales de SERVIENVIOS; las preguntas específicas son las del anteproyecto, sección "Preguntas específicas".)*

1. **¿Qué factores presentan mayor relación con los retrasos?** Completar con la lectura de las gráficas de importancia por permutación / SHAP de la sección 7.
2. **¿Qué algoritmo ofrece el mejor desempeño para tiempo de entrega (regresión)?** `MEJOR_MODELO_REGRESION` calculado arriba — completar con la magnitud de la mejora sobre la línea base y si es suficiente para validar H1.
3. **¿Qué algoritmo ofrece el mejor desempeño para incumplimiento de SLA (clasificación), priorizando recall?** `MEJOR_MODELO_CLASIFICACION` calculado arriba.
4. **¿Cuáles son las variables de mayor peso predictivo y qué lectura operativa admiten?** Completar conectando los hallazgos de SHAP/permutación con acciones concretas (p. ej. reasignar recursos en las franjas/zonas de mayor incumplimiento identificadas en la sección 2).
5. **¿En qué medida las variables externas de tráfico y clima mejoran la capacidad predictiva (H2)?** **No evaluable todavía**: la base Gold de `0300` no integra variables externas (ver su celda de "Pendiente"). Falta esa fuente de datos para poder comparar `FEATURES_REGRESION`/`FEATURES_CLASIFICACION` internas contra una versión internas+externas.

### Qué falta para que este notebook (y el proyecto) queden completos

- **Ejecutar contra datos reales.** Este notebook y `0300_CleanTransformParquet_1.ipynb` están validados funcionalmente contra datos sintéticos con el esquema real, pero nunca se han corrido contra el histórico real de SERVIENVIOS (este entorno no tiene acceso a Databricks/Drive del equipo). El equipo debe correr ambos notebooks en orden (`0300` primero, este después) y revisar que las cifras sean razonables.
- **Confirmar los supuestos marcados en `0300`**: cuál código de `ID_PROCESO` es la entrega efectiva, qué tan buena es la cobertura de `PAR.RED_OPERATIVA_OFERTA`, y si `TOTAL_ENTREGAS_DETECTADAS` es razonable frente al total de guías.
- **Variables externas de tráfico y clima** (hipótesis H2): sin fuente conectada. Es el pendiente más importante para completar el objetivo específico 2 y la pregunta específica 5.
- **Variables "de cliente"** (sección 4.3.2: tipo de cliente, prioridad/criticidad, acuerdo de nivel de servicio aplicable): no están mapeadas todavía desde `LOG.GUIAS` en `0300`.
- **Ajuste de hiperparámetros por búsqueda sistemática** (sección 4.5.3/4.5.4: "búsqueda aleatoria o por cuadrícula con validación cruzada"): este notebook entrena con hiperparámetros razonables por defecto, pero no hace la búsqueda sistemática que pide el anteproyecto; sería el siguiente paso natural una vez las cifras con datos reales confirmen que el enfoque general funciona.
- **Ajuste del umbral de decisión** de la clasificación (sección 4.5.4: "se ajustará el umbral... para priorizar la exhaustividad") — este notebook usa el umbral por defecto de 0.5; falta la curva precisión-exhaustividad para elegir un umbral operativo.
- **Redacción del documento final** (cronograma, sección 5): la parte de código de las etapas 1-5 puede quedar cubierta por `0100`→`0200`→`0300`→este notebook, pero el documento escrito del trabajo de grado es un entregable aparte que no se resuelve con notebooks.